# Car model: fine-tune the weakest brands

`car_model_predictor.ipynb` trained one small car-model classifier per brand on **frozen** features: the EfficientNet backbone never learned what separates one car model from another. That's cheap and works well for most brands (91% of test cars right), but brands with many similar models stay behind, Audi first (62%).

This notebook **fine-tunes the backbone** for those brands, one brand at a time:
- Start from the brand model's backbone plus the brand's classifier from `car_model_predictor.ipynb`, so training starts at today's accuracy instead of from zero.
- Unfreeze EfficientNet's blocks 5 to 7 (same as the brand model's fine-tuning) and train on the brand's photos from all angles, with augmentation.
- Every car model stays its own class.

**Nothing is lost if it stops.** The training state is saved after every epoch (`models/model_finetuned/backup/`); running the cell again continues from there. A finished brand is skipped.

Each brand takes a few hours on the Mac's GPU (Audi: about 15 minutes per epoch). The brands are done worst first, so stopping early still helps the brands that need it most.

In [ ]:
import json
import os
import shutil
import time
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

DATA_DIR = Path("data")
IMAGES_DIR = DATA_DIR / "resized_DVM"
MODELS_DIR = Path("models")
HEADS_DIR = MODELS_DIR / "model_heads"
FINETUNED_DIR = MODELS_DIR / "model_finetuned"
BRAND_MODEL_PATH = MODELS_DIR / "efficientnet_b0.keras"

IMAGE_SIZE = 224
BATCH_SIZE = 32
SEED = 42
FINE_TUNE_FROM = "block5a"   # first layer of block 5: blocks 5 to 7 learn
EPOCHS = 8
FINE_TUNE_BELOW = 0.90       # brands whose car accuracy is under this get fine-tuned

FINETUNED_DIR.mkdir(parents=True, exist_ok=True)

photos = pd.read_csv(MODELS_DIR / "features" / "photo_index.csv")
summary = pd.read_csv(HEADS_DIR / "summary.csv", index_col="brand")
TO_FINE_TUNE = summary[summary["car accuracy"] < FINE_TUNE_BELOW].sort_values("car accuracy").index.tolist()

print(f"GPU: {tf.config.list_physical_devices('GPU')}")
print(f"Brands to fine-tune, worst first: {', '.join(f'{b} ({summary.loc[b, "car accuracy"]:.0%})' for b in TO_FINE_TUNE)}")

## 1. Data and model

Photos load the same way as for the brand model: decoded, resized to 224×224, pixel values 0–255. Training photos get the same augmentation (shift, zoom, small rotation, contrast, brightness; no left-right flip, which would mirror badges and lettering).

The model is: augmentation → EfficientNet backbone from the brand model → average pooling → the brand's classifier from `car_model_predictor.ipynb`. That last part already knows the car models from the frozen features, so fine-tuning starts from a good place.

In [ ]:
def load_image(path, label):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE)), label


def make_dataset(part, classes, shuffle=False):
    labels = part["model"].map({m: i for i, m in enumerate(classes)}).values
    paths = (str(IMAGES_DIR) + "/" + part["path"]).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(part), seed=SEED)
    return ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def build_model(brand):
    base = keras.models.load_model(BRAND_MODEL_PATH).get_layer("efficientnetb0")
    first = next(i for i, layer in enumerate(base.layers) if layer.name.startswith(FINE_TUNE_FROM))
    base.trainable = True
    for i, layer in enumerate(base.layers):
        layer.trainable = i >= first

    head = keras.models.load_model(HEADS_DIR / f"{brand}.keras")
    augment = keras.Sequential([
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomZoom(0.15),
        layers.RandomRotation(0.03),
        layers.RandomContrast(0.2),
        layers.RandomBrightness(0.2),
    ], name="augment")

    inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    x = base(augment(inputs), training=False)   # BatchNormalization stays in prediction mode
    x = layers.GlobalAveragePooling2D()(x)
    outputs = head(x)
    model = keras.Model(inputs, outputs, name=f"{brand.replace(' ', '_')}_car_models")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )
    return model

## 2. Fine-tune, brand by brand

For each brand, worst first: train for up to 8 epochs, keep the best epoch by validation loss (`models/model_finetuned/<brand>.keras`), stop early after 2 epochs without improvement. The first epoch's progress bar shows how long an epoch takes.

Run this cell again after an interruption: finished brands are skipped, and the brand that was training resumes from its last finished epoch.

In [ ]:
for brand in TO_FINE_TUNE:
    done_marker = FINETUNED_DIR / f"{brand}_done.txt"
    if done_marker.exists():
        print(f"{brand}: already fine-tuned, skipping")
        continue

    classes = json.loads((HEADS_DIR / f"{brand}_classes.json").read_text())
    part = photos[photos["brand"] == brand]
    train, val = part[part["split"] == "train"], part[part["split"] == "val"]
    counts = train["model"].value_counts()
    class_weight = {i: len(train) / (len(classes) * counts.get(m, 1)) for i, m in enumerate(classes)}

    log_path = FINETUNED_DIR / f"{brand}_log.csv"
    best_so_far = pd.read_csv(log_path)["val_loss"].min() if log_path.exists() else None
    print(f"\n{brand}: {len(classes)} car models, {len(train):,} training photos, {len(val):,} validation photos", flush=True)

    model = build_model(brand)
    model.fit(
        make_dataset(train, classes, shuffle=True),
        validation_data=make_dataset(val, classes),
        epochs=EPOCHS,
        class_weight=class_weight,
        callbacks=[
            keras.callbacks.BackupAndRestore(str(FINETUNED_DIR / "backup" / brand)),
            keras.callbacks.CSVLogger(log_path, append=True),
            keras.callbacks.ModelCheckpoint(
                FINETUNED_DIR / f"{brand}.keras", monitor="val_loss", save_best_only=True,
                initial_value_threshold=best_so_far,
            ),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=1),
            keras.callbacks.EarlyStopping(monitor="val_loss", patience=2),
        ],
    )
    (FINETUNED_DIR / f"{brand}_classes.json").write_text(json.dumps(classes))
    done_marker.write_text("done")
    print(f"{brand}: saved {FINETUNED_DIR / (brand + '.keras')}", flush=True)

## 3. Before and after

Test cars of each fine-tuned brand, photo probabilities averaged per car (like the dashboard): car-model accuracy with frozen features (`car_model_predictor.ipynb`) against the fine-tuned model. Saved to `models/model_finetuned/comparison.csv`.

In [ ]:
rows = []
for brand in TO_FINE_TUNE:
    path = FINETUNED_DIR / f"{brand}.keras"
    if not path.exists():
        continue
    classes = json.loads((HEADS_DIR / f"{brand}_classes.json").read_text())
    test = photos[(photos["brand"] == brand) & (photos["split"] == "test")]
    model = keras.models.load_model(path)
    probabilities = tf.nn.softmax(model.predict(make_dataset(test, classes), verbose=0)).numpy()

    photo_correct = np.array(classes)[probabilities.argmax(axis=1)] == test["model"].values
    per_car = pd.DataFrame(probabilities, index=test["advert_id"].values).groupby(level=0).mean()
    truth = test.groupby("advert_id")["model"].first().loc[per_car.index]
    car_correct = np.array(classes)[per_car.values.argmax(axis=1)] == truth.values

    rows.append({
        "brand": brand,
        "car accuracy before": summary.loc[brand, "car accuracy"],
        "car accuracy after": car_correct.mean(),
        "photo accuracy before": summary.loc[brand, "photo accuracy"],
        "photo accuracy after": photo_correct.mean(),
        "test cars": len(per_car),
    })

comparison = pd.DataFrame(rows).set_index("brand")
comparison.to_csv(FINETUNED_DIR / "comparison.csv")
comparison.style.format({c: "{:.1%}" for c in comparison.columns if "accuracy" in c})

In [ ]:
if len(comparison):
    ax = comparison[["car accuracy before", "car accuracy after"]].plot.barh(figsize=(8, 1 + 0.6 * len(comparison)), width=0.8)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Car-model accuracy on test cars")
    ax.set_title("Frozen features vs fine-tuned")
    plt.tight_layout()
    plt.show()